In [0]:
DROP VIEW IF EXISTS bootcamp_matias.gold.Kpi_propiedades;

CREATE VIEW bootcamp_matias.gold.kpi_propiedades
COMMENT 'KPI Generales de Propiedades'
AS
WITH metricas (
SELECT
 
  COUNT(*) AS total_propiedades,

  COUNT( DISTINCT B.ciudad) AS total_ciudades,
  
  --Tipo de operaciones
  COUNT(CASE WHEN C.tipo_operacion = 'venta' THEN 1 END ) AS en_venta,
  COUNT(CASE WHEN C.tipo_operacion = 'alquiler' THEN 1 END ) AS total_alquileres,

  --Promedio Generales
  ROUND(AVG(A.ambientes), 2) AS ambiente_promedio,
  ROUND(AVG(A.expensas),2) AS expensa_promedio,
  ROUND(AVG(A.precio),2) AS precio_promedio,
  ROUND(AVG(A.precio_por_m2), 2) AS precio_m2_promedio

FROM bootcamp_matias.gold.fact_propiedades A

LEFT JOIN bootcamp_matias.gold.dim_zona B 
  ON A.zona_id = B.zona_id

LEFT JOIN bootcamp_matias.gold.dim_tipo_operacion C
  ON A.tipo_operacion_id = C.tipo_operacion_id

WHERE 
  moneda = 'ARS'
  AND precio > 0
  AND precio_por_m2 > 0
  AND ambientes IS NOT NULL
)
SELECT
  total_propiedades,
  total_ciudades,
  en_venta,
  ROUND(en_venta *100.0 / total_propiedades,2) AS pct_ventas,
  total_alquileres,
  ROUND(total_alquileres * 100.0 / total_propiedades,2) AS pct_alquiler,
  ambiente_promedio,
  expensa_promedio,
  precio_m2_promedio,
  CURRENT_TIMESTAMP() AS _refresh_timestamp
FROM metricas